# Demo: Tablero de Ajedrez

Demostración paso a paso: movimientos, capturas y validación de las clases `Board` y piezas.

**Convención visual:** Mayúsculas = blancas (filas 0-1) | minúsculas = negras (filas 6-7)

| Símbolo | Pieza   |
|---------|---------|
| P / p   | Peón    |
| C / c   | Caballo |
| A / a   | Alfil   |
| T / t   | Torre   |
| R / r   | Rey     |
| Q / q   | Reina   |

## Setup

In [ ]:
import sys
from pathlib import Path

# Detectar la carpeta elements sin importar desde dónde se lanzó Jupyter
for _candidate in [
    Path('src/parser/elements'),   # desde la raíz del proyecto
    Path('../elements'),            # desde src/parser/notebooks/
    Path('elements'),              # desde src/parser/
]:
    if _candidate.resolve().exists():
        sys.path.insert(0, str(_candidate.resolve()))
        print(f'✓ Path añadido: {_candidate.resolve()}')
        break
else:
    raise RuntimeError('No se encontró la carpeta elements. Revisá desde dónde se lanzó Jupyter.')

from board import Board
from pieces import Pawn, Knight, Bishop, Rook, King, Queen

In [ ]:
LETRA = {Pawn: 'P', Knight: 'C', Bishop: 'A', Rook: 'T', King: 'R', Queen: 'Q'}

def mostrar_tablero(piezas, titulo=''):
    if titulo:
        print(f'\n=== {titulo} ===')
    print('    0  1  2  3  4  5  6  7')
    print('  +' + '--+' * 8)
    for i, fila in enumerate(piezas):
        row = f'{i} |'
        for pz in fila:
            if pz is None:
                row += ' .|'
            else:
                l = LETRA[type(pz)]
                row += f" {l if pz.color == 'blanco' else l.lower()}|"
        print(row)
        print('  +' + '--+' * 8)
    print()

## 1. Posición inicial
Blancas arriba (filas 0-1), negras abajo (filas 6-7).

In [ ]:
b = Board(nueva_partida=True)
p = b.piezas

mostrar_tablero(p, 'Posición inicial')
print('Matriz numérica (+ blancas / - negras):')
print(b.matriz)

### Detalle de piezas en posición inicial

In [ ]:
print('--- Fila 0: back rank blancas ---')
for j in range(8):
    print(f'  [0][{j}] = {p[0][j]}')

print('\n--- Fila 1: peones blancos ---')
for j in range(8):
    print(f'  [1][{j}] = {p[1][j]}')

print('\n--- Fila 6: peones negros ---')
for j in range(8):
    print(f'  [6][{j}] = {p[6][j]}')

print('\n--- Fila 7: back rank negras ---')
for j in range(8):
    print(f'  [7][{j}] = {p[7][j]}')

## 2. Peones — avance inicial
Blanco avanza hacia filas crecientes (+1), negro hacia decrecientes (-1).

In [ ]:
peon_b = p[1][4]
print(f'Pieza: {peon_b}')
print(f'primer_mov = {peon_b.primer_mov}')
print(f'Movimientos válidos: {peon_b.obtener_movimientos_validos(p)}')
print('  → puede avanzar 1 o 2 casillas (primer movimiento)')

peon_b.mover(3, 4, p)
print(f'\n→ Movido a [3][4]: {peon_b}')
print(f'  primer_mov ahora = {peon_b.primer_mov}')
mostrar_tablero(p, 'Peón blanco e2→e4')

In [ ]:
peon_n = p[6][3]
print(f'Pieza: {peon_n}')
print(f'Movimientos válidos: {peon_n.obtener_movimientos_validos(p)}')

peon_n.mover(4, 3, p)
print(f'\n→ Movido a [4][3]: {peon_n}')
mostrar_tablero(p, 'Peón negro d7→d5')

## 3. Movimiento inválido
`mover()` lanza `ValueError` si el destino no está en `obtener_movimientos_validos()`.

In [ ]:
print('Intentando retroceder el peón blanco [3][4] → [2][4]...')
try:
    p[3][4].mover(2, 4, p)
except ValueError as e:
    print(f'  ✗ Error: {e}')

print('\nIntentando moverlo lateralmente [3][4] → [3][5]...')
try:
    p[3][4].mover(3, 5, p)
except ValueError as e:
    print(f'  ✗ Error: {e}')

print('\nIntentando moverlo 3 casillas adelante [3][4] → [6][4] (no puede, ya no es primer mov)...')
try:
    p[3][4].mover(6, 4, p)
except ValueError as e:
    print(f'  ✗ Error: {e}')

## 4. Captura con peón
El peón captura **en diagonal** (nunca recto). El peón blanco en `[3][4]` ve al peón negro en `[4][3]`.

In [ ]:
movs = p[3][4].obtener_movimientos_validos(p)
print(f'Peón blanco: {p[3][4]}')
print(f'Movimientos válidos: {movs}')
print('  (4,4) = avance recto | (4,3) = captura diagonal')

print(f'\nCasilla [4][3] contiene: {p[4][3]}')
print('\n--- capturando ---')
comida = p[3][4].mover(4, 3, p)
print(f'Pieza comida: {comida}')
print(f'Peón blanco ahora en: {p[4][3]}')
mostrar_tablero(p, 'Peón blanco captura en d5')

## 5. Caballo — movimiento en L
Salta en L, puede pasar por encima de piezas. 8 destinos posibles desde el centro.

In [ ]:
cb = p[0][6]
print(f'Caballo blanco: {cb}')
print(f'Movimientos válidos: {cb.obtener_movimientos_validos(p)}')

cb.mover(2, 5, p)
print(f'\n→ Caballo blanco movido a [2][5]')
mostrar_tablero(p, 'Caballo blanco Cf3')

In [ ]:
cn = p[7][6]
print(f'Caballo negro: {cn}')
print(f'Movimientos válidos: {cn.obtener_movimientos_validos(p)}')

cn.mover(5, 5, p)
print(f'\n→ Caballo negro movido a [5][5]')
mostrar_tablero(p, 'Caballo negro Cf6')

In [ ]:
print(f'Caballo blanco [2][5] avanza:')
print(f'  Movimientos válidos: {p[2][5].obtener_movimientos_validos(p)}')

p[2][5].mover(4, 4, p)
print('→ Caballo blanco movido a [4][4]')
mostrar_tablero(p, 'Caballo blanco Ce5')

In [ ]:
print(f'Caballo negro [5][5]:')
movs = p[5][5].obtener_movimientos_validos(p)
print(f'  Movimientos válidos: {movs}')
for r, c in movs:
    obj = p[r][c]
    if obj is not None and obj.color != 'blanco':
        pass  # mismo color que el caballo negro
    elif obj is not None:
        print(f'  ¡Puede comer {obj} en [{r}][{c}]!')

print('\n--- Caballo negro captura peón blanco en [4][3] ---')
comida = p[5][5].mover(4, 3, p)
print(f'Pieza comida: {comida}')
mostrar_tablero(p, 'Caballo negro captura peón d5')

In [ ]:
print('Caballo blanco [4][4]:')
movs = p[4][4].obtener_movimientos_validos(p)
print(f'  Movimientos válidos: {movs}')
for r, c in movs:
    obj = p[r][c]
    if obj is not None and obj.color != 'blanco':
        print(f'  ¡Puede comer {obj} en [{r}][{c}]!')

print('\n--- Caballo blanco captura peón negro en [6][5] ---')
comida = p[4][4].mover(6, 5, p)
print(f'Pieza comida: {comida}')
mostrar_tablero(p, 'Caballo blanco captura peón f7')

In [ ]:
print('Caballo blanco ahora en [6][5] — profundidad máxima:')
movs = p[6][5].obtener_movimientos_validos(p)
print(f'  Movimientos válidos: {movs}')
for r, c in movs:
    obj = p[r][c]
    if obj is not None and obj.color != 'blanco':
        print(f'  ¡Puede comer {obj} en [{r}][{c}]!')

print('\n--- Caballo blanco captura Reina negra en [7][3] ---')
comida = p[6][5].mover(7, 3, p)
print(f'Pieza comida: {comida}')
mostrar_tablero(p, 'Caballo blanco captura Reina negra!')

## 6. Alfil — deslizamiento diagonal
Se mueve diagonalmente cualquier cantidad de casillas hasta chocar con una pieza.

In [ ]:
alfil_b = p[0][5]
print(f'Alfil blanco: {alfil_b}')
movs = alfil_b.obtener_movimientos_validos(p)
print(f'Movimientos válidos: {movs}')
print('  El camino (0,5)→(1,4)→(2,3)→... está libre porque el peón e2 se movió')

alfil_b.mover(3, 2, p)
print(f'\n→ Alfil movido a [3][2]')
mostrar_tablero(p, 'Alfil blanco Ad3')

In [ ]:
print('Alfil blanco [3][2]:')
movs = p[3][2].obtener_movimientos_validos(p)
print(f'  Movimientos válidos: {movs}')
for r, c in movs:
    obj = p[r][c]
    if obj is not None and obj.color != 'blanco':
        print(f'  ¡Puede comer {obj} en [{r}][{c}]!')

print('\n--- Alfil captura Caballo negro en [4][3] ---')
comida = p[3][2].mover(4, 3, p)
print(f'Pieza comida: {comida}')
mostrar_tablero(p, 'Alfil captura Caballo negro d5')

In [ ]:
print('Alfil blanco [4][3]:')
movs = p[4][3].obtener_movimientos_validos(p)
print(f'  Movimientos válidos: {movs}')
for r, c in movs:
    obj = p[r][c]
    if obj is not None and obj.color != 'blanco':
        print(f'  ¡Puede comer {obj} en [{r}][{c}]!')

print('\n--- Alfil captura peón negro en [6][1] ---')
comida = p[4][3].mover(6, 1, p)
print(f'Pieza comida: {comida}')
mostrar_tablero(p, 'Alfil captura peón negro b7')

## 7. Torre — movimiento horizontal y vertical
Usamos un segundo tablero `b2` y abrimos la columna `a`.

In [ ]:
b2 = Board(nueva_partida=True)
p2 = b2.piezas

p2[1][0].mover(3, 0, p2)  # peón a2→a4
p2[6][0].mover(4, 0, p2)  # peón negro a7→a5

mostrar_tablero(p2, 'Tablero b2 — col a abierta')

In [ ]:
torre_b = p2[0][0]
print(f'Torre blanca: {torre_b}')
print(f'Movimientos válidos: {torre_b.obtener_movimientos_validos(p2)}')

torre_b.mover(2, 0, p2)
print(f'\n→ Torre movida a [2][0]')
mostrar_tablero(p2, 'Torre blanca Ta3')

In [ ]:
print('Torre blanca [2][0] — movimiento horizontal:')
movs = p2[2][0].obtener_movimientos_validos(p2)
print(f'  Movimientos válidos: {sorted(movs)}')

p2[2][0].mover(2, 6, p2)
print('\n→ Torre deslizada a [2][6]')
mostrar_tablero(p2, 'Torre Ta3→Tg3')

In [ ]:
print('Torre blanca [2][6] — movimiento vertical:')
movs = p2[2][6].obtener_movimientos_validos(p2)
print(f'  Movimientos válidos: {sorted(movs)}')
for r, c in movs:
    obj = p2[r][c]
    if obj is not None and obj.color != 'blanco':
        print(f'  ¡Puede comer {obj} en [{r}][{c}]!')

print('\n--- Torre captura peón negro en [6][6] ---')
comida = p2[2][6].mover(6, 6, p2)
print(f'Pieza comida: {comida}')
mostrar_tablero(p2, 'Torre captura peón negro g7')

## 8. Reina — todas las direcciones
Combina torre + alfil: 8 direcciones deslizantes.

In [ ]:
# Abrir más caminos para la reina en b2
p2[1][3].mover(3, 3, p2)  # d2→d4
p2[1][4].mover(3, 4, p2)  # e2→e4

reina_b = p2[0][3]
print(f'Reina blanca: {reina_b}')
movs = reina_b.obtener_movimientos_validos(p2)
print(f'Movimientos válidos ({len(movs)} posiciones):')
print(f'  {sorted(movs)}')

In [ ]:
# Mover reina en diagonal hasta [4][7]
p2[0][3].mover(4, 7, p2)
print('→ Reina movida a [4][7]')
mostrar_tablero(p2, 'Reina blanca Qh5')

movs2 = p2[4][7].obtener_movimientos_validos(p2)
print(f'Movimientos válidos desde [4][7] ({len(movs2)} posiciones):')
print(f'  {sorted(movs2)}')

In [ ]:
for r, c in p2[4][7].obtener_movimientos_validos(p2):
    obj = p2[r][c]
    if obj is not None and obj.color != 'blanco':
        print(f'  Puede comer {obj} en [{r}][{c}]')

print('\n--- Reina captura peón negro en [6][5] ---')
comida = p2[4][7].mover(6, 5, p2)
print(f'Pieza comida: {comida}')
mostrar_tablero(p2, 'Reina captura peón negro f7')

## 9. En Passant

Captura especial del peón: cuando un peón enemigo avanza **dos casillas** y queda al lado del nuestro,
podemos capturarlo **como si hubiera avanzado solo una** — yendo a la casilla que *pasó por alto*.

**Regla clave:** solo es válido en el turno inmediatamente siguiente al avance doble.

La clase recibe `en_passant_col` como parámetro opcional (responsabilidad de la partida registrar cuándo aplica).

In [ ]:
# Tablero nuevo para la demo
b3 = Board(nueva_partida=True)
p3 = b3.piezas

# 1. Peon blanco avanza doble (e2->e4)
p3[1][4].mover(3, 4, p3)
# 2. Peon negro hace un mov cualquiera (h7->h6)
p3[6][7].mover(5, 7, p3)
# 3. Peon blanco avanza uno mas (e4->e5) — llega a fila 4
p3[3][4].mover(4, 4, p3)

mostrar_tablero(p3, 'Antes del avance doble negro')

In [ ]:
# 4. Peon negro avanza doble (f7->f5): queda en fila 4, col 5 — al lado del blanco
p3[6][5].mover(4, 5, p3)

# La partida registra que col 5 acaba de tener un avance doble
en_passant_col = 5

mostrar_tablero(p3, 'Peon negro f7->f5 (avance doble) — en passant disponible')
print(f'Peon blanco: {p3[4][4]}')
print(f'Peon negro:  {p3[4][5]}')
print(f'en_passant_col = {en_passant_col}  (col del peon que acabo de avanzar doble)')

In [ ]:
# Comparar movimientos con y sin en_passant_col
movs_sin = p3[4][4].obtener_movimientos_validos(p3)
movs_con = p3[4][4].obtener_movimientos_validos(p3, en_passant_col=en_passant_col)

print(f'Movimientos SIN en_passant_col: {movs_sin}')
print(f'Movimientos CON en_passant_col: {movs_con}')
print()
print('  (5,4) = avance recto normal')
print('  (5,5) = en passant  <- aparece SOLO cuando se pasa en_passant_col=5')

In [ ]:
# La casilla destino (5,5) esta vacia — el peon negro esta en (4,5)
print(f'Casilla destino [5][5]: {p3[5][5]}  <- vacia')
print(f'Peon negro en    [4][5]: {p3[4][5]}  <- aca esta')
print()
print('--- ejecutando en passant ---')
comida = p3[4][4].mover(5, 5, p3, en_passant_col=en_passant_col)
print(f'Pieza comida: {comida}')
print(f'Peon blanco ahora en [5][5]: {p3[5][5]}')
print(f'Casilla [4][5] tras captura: {p3[4][5]}  <- eliminada')
print(f'Casilla [4][4] (origen):     {p3[4][4]}  <- liberada')
mostrar_tablero(p3, 'Despues del en passant — peon negro capturado en [4][5]')

In [ ]:
# Sin en_passant_col el movimiento es rechazado (la ventana se cerro)
print('Intentando el mismo movimiento SIN pasar en_passant_col...')
b4 = Board(nueva_partida=True)
p4 = b4.piezas
p4[1][4].mover(3, 4, p4)
p4[6][7].mover(5, 7, p4)
p4[3][4].mover(4, 4, p4)
p4[6][5].mover(4, 5, p4)
try:
    p4[4][4].mover(5, 5, p4)  # sin en_passant_col
except ValueError as e:
    print(f'  Error (correcto): {e}')

## 10. Enroque

Movimiento especial donde el **rey se desplaza 2 columnas** y la torre salta a su lado.

**Condiciones requeridas:**
- Ni el rey ni la torre involucrada se han movido antes (`primer_mov = True`)
- Las casillas entre ambos están vacías
- (Nota: no se verifica jaque en esta implementación)

| Tipo | Rey va a | Torre va a |
|------|----------|------------|
| Corto (kingside)  | col 6 | col 5 |
| Largo (queenside) | col 2 | col 3 |

### Enroque corto (kingside)
Hay que despejar las casillas `[0][5]` (alfil) y `[0][6]` (caballo) antes de enrocar.

In [ ]:
b_ek = Board(nueva_partida=True)
p_ek = b_ek.piezas

# Despejar camino: mover peón e2→e4, luego alfil y caballo salen
p_ek[1][4].mover(3, 4, p_ek)   # peón e2→e4  (abre diagonal del alfil)
p_ek[0][5].mover(2, 3, p_ek)   # Alfil f1→d3
p_ek[0][6].mover(2, 5, p_ek)   # Caballo g1→f3

mostrar_tablero(p_ek, 'Camino despejado para enroque corto')
print(f'Rey   [0][4] primer_mov = {p_ek[0][4].primer_mov}')
print(f'Torre [0][7] primer_mov = {p_ek[0][7].primer_mov}')
print(f'[0][5] vacía: {p_ek[0][5] is None}')
print(f'[0][6] vacía: {p_ek[0][6] is None}')

In [ ]:
movs_rey = p_ek[0][4].obtener_movimientos_validos(p_ek)
print(f'Movimientos válidos del rey: {movs_rey}')
print()
print('  (0,3) = mover izquierda normal')
print('  (0,5) = mover derecha normal')
print('  (1,4) = diagonal (casilla libre)')
print('  (0,6) = ENROQUE CORTO  ← rey salta 2 columnas')

In [ ]:
print('--- ejecutando enroque corto: rey [0][4] → [0][6] ---')
p_ek[0][4].mover(0, 6, p_ek)

mostrar_tablero(p_ek, 'Después del enroque corto')
print(f'Rey   en [0][6]: {p_ek[0][6]}  (primer_mov={p_ek[0][6].primer_mov})')
print(f'Torre en [0][5]: {p_ek[0][5]}  (primer_mov={p_ek[0][5].primer_mov})')
print(f'[0][4] (origen rey) vacía  : {p_ek[0][4] is None}')
print(f'[0][7] (origen torre) vacía: {p_ek[0][7] is None}')

### Enroque largo (queenside)
Hay que despejar `[0][1]` (caballo), `[0][2]` (alfil) y `[0][3]` (reina) para enrocar con la torre de `[0][0]`.

In [ ]:
b_eq = Board(nueva_partida=True)
p_eq = b_eq.piezas

# Sacar caballo b1
p_eq[0][1].mover(2, 2, p_eq)   # Cb1→c3

# Sacar reina (d1→d2): antes hay que abrir la columna d con el peón
p_eq[1][3].mover(3, 3, p_eq)   # peón d2→d4
p_eq[0][3].mover(1, 3, p_eq)   # Reina d1→d2

# Sacar alfil c1 por la diagonal (necesita abrir b2 primero)
p_eq[1][1].mover(3, 1, p_eq)   # peón b2→b4
p_eq[0][2].mover(1, 1, p_eq)   # Alfil c1→b2

mostrar_tablero(p_eq, 'Camino despejado para enroque largo')
print(f'[0][1] vacía: {p_eq[0][1] is None}')
print(f'[0][2] vacía: {p_eq[0][2] is None}')
print(f'[0][3] vacía: {p_eq[0][3] is None}')
print(f'Rey   [0][4] primer_mov = {p_eq[0][4].primer_mov}')
print(f'Torre [0][0] primer_mov = {p_eq[0][0].primer_mov}')

In [ ]:
movs_rey_q = p_eq[0][4].obtener_movimientos_validos(p_eq)
print(f'Movimientos válidos del rey: {movs_rey_q}')
print()
print('  (0,3) = mover izquierda normal')
print('  (0,5) = mover derecha normal')
print('  (0,2) = ENROQUE LARGO  ← rey salta 2 columnas hacia la izquierda')

print()
print('--- ejecutando enroque largo: rey [0][4] → [0][2] ---')
p_eq[0][4].mover(0, 2, p_eq)

mostrar_tablero(p_eq, 'Después del enroque largo')
print(f'Rey   en [0][2]: {p_eq[0][2]}  (primer_mov={p_eq[0][2].primer_mov})')
print(f'Torre en [0][3]: {p_eq[0][3]}  (primer_mov={p_eq[0][3].primer_mov})')
print(f'[0][4] (origen rey) vacía  : {p_eq[0][4] is None}')
print(f'[0][0] (origen torre) vacía: {p_eq[0][0] is None}')

### Enroque bloqueado
El enroque desaparece de los movimientos válidos si el rey **o** la torre ya se movieron (`primer_mov = False`).

In [ ]:
b_bl = Board(nueva_partida=True)
p_bl = b_bl.piezas

# Despejar el camino (igual que enroque corto)
p_bl[1][4].mover(3, 4, p_bl)
p_bl[0][5].mover(2, 3, p_bl)
p_bl[0][6].mover(2, 5, p_bl)

print('=== Caso 1: torre ya se movió ===')
p_bl[0][7].mover(0, 6, p_bl)  # torre va a g1 (se mueve una vez)
p_bl[0][6].mover(0, 7, p_bl)  # torre vuelve a h1
print(f'Torre [0][7] primer_mov = {p_bl[0][7].primer_mov}  <- ya se movió')
movs = p_bl[0][4].obtener_movimientos_validos(p_bl)
print(f'Movimientos del rey: {movs}')
print('  → (0,6) NO aparece: enroque corto bloqueado')

print()
print('=== Caso 2: rey ya se movió ===')
b_bl2 = Board(nueva_partida=True)
p_bl2 = b_bl2.piezas
p_bl2[1][4].mover(3, 4, p_bl2)
p_bl2[0][5].mover(2, 3, p_bl2)
p_bl2[0][6].mover(2, 5, p_bl2)
p_bl2[0][4].mover(0, 5, p_bl2)  # rey se mueve a f1
p_bl2[0][5].mover(0, 4, p_bl2)  # rey vuelve a e1
print(f'Rey [0][4] primer_mov = {p_bl2[0][4].primer_mov}  <- ya se movió')
movs2 = p_bl2[0][4].obtener_movimientos_validos(p_bl2)
print(f'Movimientos del rey: {movs2}')
print('  → ningún enroque disponible')

## 11. Promoción del peón

Cuando un peón llega a la **última fila del bando rival** (fila 7 para blancas, fila 0 para negras) se transforma en otra pieza.

`mover()` acepta el parámetro `promocion=<Clase>`. Si se omite, **promociona a Reina** por defecto (la opción casi siempre óptima).

| Parámetro | Resultado |
|-----------|-----------|
| `promocion=None` (default) | → `Queen` |
| `promocion=Knight` | subpromoción a Caballo |
| `promocion=Rook` | subpromoción a Torre |
| `promocion=Bishop` | subpromoción a Alfil |

### Promoción a Reina (por defecto)

In [ ]:
b_prom = Board(nueva_partida=True)
p_prom = b_prom.piezas

# Colocamos el peón blanco directamente en fila 6 para demo rápida
p_prom[6][0] = None     # liberar casilla (tenía peón negro)
p_prom[1][0] = None     # quitar peón blanco original de a2
p_prom[7][0] = None     # liberar fila final (tenía torre negra)

peon_prom = Pawn('blanco', 6, 0)
p_prom[6][0] = peon_prom

mostrar_tablero(p_prom, 'Peón blanco en fila 6, listo para promover')
print(f'Peón: {peon_prom}')
print(f'Movimientos disponibles: {peon_prom.obtener_movimientos_validos(p_prom)}')

In [ ]:
print('--- promoviendo sin especificar clase (default = Reina) ---')
peon_prom.mover(7, 0, p_prom)   # sin parámetro promocion

print()
print(f'[7][0] ahora es: {p_prom[7][0]}')
print(f'tipo            : {type(p_prom[7][0]).__name__}')
print(f'[6][0] (origen) : {p_prom[6][0]}  ← vacía')
mostrar_tablero(p_prom, 'Después de la promoción a Reina')

### Subpromoción a Caballo
A veces promover a Caballo es mejor (ej. el Caballo puede dar jaque de inmediato). Se pasa `promocion=Knight`.

In [ ]:
b_sub = Board(nueva_partida=True)
p_sub = b_sub.piezas

# Peón blanco en fila 6 col 3; [7][3]=reina negra (bloquea avance), [7][4]=rey negro
p_sub[6][3] = None    # liberar d7 (había peón negro)
p_sub[1][3] = None    # quitar peón blanco d2

peon_sub = Pawn('blanco', 6, 3)
p_sub[6][3] = peon_sub

print(f'Peón: {peon_sub}')
print(f'Movimientos: {peon_sub.obtener_movimientos_validos(p_sub)}')
print(f'  (7,3) bloqueado por Reina negra | (7,2) captura Alfil | (7,4) captura Rey')
print()

print('--- promoviendo a Caballo capturando el Rey negro en [7][4] ---')
comida = peon_sub.mover(7, 4, p_sub, promocion=Knight)
print(f'Pieza capturada : {comida}')
print(f'[7][4] ahora es : {p_sub[7][4]}  tipo={type(p_sub[7][4]).__name__}')
mostrar_tablero(p_sub, 'Subpromoción a Caballo — capturando al Rey negro!')

## Estado final — tablero principal (b)
Resumen de todas las capturas realizadas durante la demo.

In [ ]:
mostrar_tablero(p, 'Estado final tablero b')

# Reconstruir la matriz numérica desde el estado actual de piezas
import numpy as np
VALOR = {Pawn: 1, Knight: 2, Bishop: 3, Rook: 4, King: 5, Queen: 6}
mat = np.zeros((8, 8), dtype=int)
for i in range(8):
    for j in range(8):
        pz = p[i][j]
        if pz is not None:
            mat[i][j] = VALOR[type(pz)] * (1 if pz.color == 'blanco' else -1)

print('Matriz numérica actual:')
print(mat)